# 04 — MediaPipe Hands sanity check

**Question.** Does MediaPipe Hands reliably detect hand landmarks on our existing thesis footage?

This is the **Phase 1** of the gesture module (`docs/GESTURE_MODULE.md`). We are *not* training a classifier yet — only confirming that the keypoint extractor works on our distance / lighting / framing.

**Outputs (next to the source video):**

* `<name>_overlay.mp4` — video with hand skeletons drawn on every frame.
* `<name>_keypoints.csv` — per-frame, per-hand, 21 × 3 landmarks.

If MediaPipe fails on a given clip (subject too far, too small in frame, etc.), we log that for the *data collection* plan in Phase 2.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from gesture.mediapipe_overlay import run as run_mediapipe

## 1. Pick a clip and run

Edit `VIDEO` below to point at one of your existing thesis videos. The script writes both an overlay video and a CSV with one row per detected hand per frame.

In [ ]:
VIDEO = Path('/Users/simayyalcin/Desktop/uav_perception/raw_videos/01.mp4')   # ← edit me
OUT_PREFIX = REPO_ROOT / 'data' / 'gesture' / VIDEO.stem

if VIDEO.exists():
    overlay_path, keypoints_path = run_mediapipe(
        video_path=VIDEO,
        output_prefix=OUT_PREFIX,
        max_hands=2,
        detection_confidence=0.5,
        tracking_confidence=0.5,
    )
else:
    print(f'Video not found: {VIDEO}\nEdit the VIDEO variable above.')

## 2. Detection summary

In [ ]:
KEYPOINTS_CSV = OUT_PREFIX.with_name(OUT_PREFIX.name + '_keypoints.csv')

if KEYPOINTS_CSV.exists():
    kp = pd.read_csv(KEYPOINTS_CSV)
    print(f'Total hand instances detected: {len(kp)}')
    print(f'Frames with detection:         {kp["frame"].nunique()}')
    print(f'Mean detection score:          {kp["score"].mean():.3f}')
    print('Handedness counts:')
    print(kp['handedness'].value_counts())
    display(kp.head())
else:
    print('Run the previous cell first.')

## 3. Plot a wrist trajectory over time

`kp0_x` / `kp0_y` are the wrist landmark (in normalised image coordinates ∈ [0, 1]). Seeing it move smoothly is a quick sanity check.

In [ ]:
if KEYPOINTS_CSV.exists():
    kp = pd.read_csv(KEYPOINTS_CSV)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for hand_idx, color in zip([0, 1], ['tab:blue', 'tab:orange']):
        sub = kp[kp['hand_index'] == hand_idx]
        if not len(sub):
            continue
        axes[0].plot(sub['time_seconds'], sub['kp0_x'], color=color, label=f'hand {hand_idx}')
        axes[1].plot(sub['time_seconds'], sub['kp0_y'], color=color, label=f'hand {hand_idx}')
    axes[0].set_title('Wrist X over time'); axes[0].set_xlabel('time (s)'); axes[0].set_ylabel('normalised X')
    axes[1].set_title('Wrist Y over time'); axes[1].set_xlabel('time (s)'); axes[1].set_ylabel('normalised Y')
    for ax in axes: ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

## 4. Show a sample frame with the overlay

Reads one frame from the overlay video for a quick visual check.

In [ ]:
OVERLAY = OUT_PREFIX.with_name(OUT_PREFIX.name + '_overlay.mp4')

if OVERLAY.exists():
    import cv2
    cap = cv2.VideoCapture(str(OVERLAY))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    # Pick a frame in the middle (more likely to have action than the first frame).
    cap.set(cv2.CAP_PROP_POS_FRAMES, n // 2)
    ok, frame = cap.read()
    cap.release()
    if ok:
        plt.figure(figsize=(10, 7))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(f'Frame {n//2} of {OVERLAY.name}')
        plt.show()
    else:
        print('Could not read a frame from the overlay video.')
else:
    print('Overlay video not found:', OVERLAY)

## 5. Findings (fill in)

* Detection rate on this clip: <X %> of frames had ≥ 1 hand detected.
* Mean confidence score: <Y>.
* Failure modes observed: <e.g. "loses hands when subject is > 4 m away">.
* Next step: decide whether to (a) train classifier on existing public data (Phase 2 of `GESTURE_MODULE.md`) or (b) record a small custom dataset first.